# Week 8 — Solved

Narrative data visualization (Segel & Heer sections 4–6) and the micro-project data story.

## Exercise 1.1 — Segel & Heer taxonomy questions

**Purpose of Figure 7:** It gives designers a structured vocabulary for their choices, organized into *Visual Narrative* (how individual charts are designed) and *Narrative Structure* (how charts are sequenced and organized into a story). It makes design decisions explicit and comparable across different visualization pieces.

**Most common design choices:**
- *Visual Narrative / Visual Structuring:* Consistent visual platform (same color scheme, layout across frames)
- *Visual Narrative / Highlighting:* Feature distinction (color or size used to call out key data points)
- *Narrative Structure / Ordering:* User-directed paths
- *Narrative Structure / Interactivity:* Hover highlighting and filtering

**Rarely used choices:** 'Progress bar' and 'Stimulating default views' — possibly because designers assume readers will start from the beginning, and because setting up compelling defaults requires extra effort.

**Favorite genre:** *Annotated chart* — single chart with text annotations. It's simple to produce, requires no special technology, and forces the author to commit to a specific interpretation. Journalists use it because it works in both print and digital.

**Least favorite:** *Partitioned poster* — splits a single topic across many panels arranged spatially. Hard to reproduce on mobile and requires a lot of screen space.

## Exercise 1.2 — Applying taxonomy to Week 6 visualizations

**Interactive hourly bar chart (Ex 2.1):**
- *Visual Narrative:* Consistent visual platform ✓, no highlighting (user-controlled), no annotations
- *Narrative Structure:* User-directed ordering, filtering via legend toggle, details on demand via hover
- Position: strongly reader-driven — closer to 'free-form exploration'

**Animated line chart (Ex 2.2):**
- *Visual Narrative:* Consistent visual platform ✓, fixed y-axis ✓ (prevents misleading rescaling)
- *Narrative Structure:* Linear ordering (year by year), author-controlled animation
- Position: slightly more author-driven — the year sequence is imposed, but the user can pause/rewind

**To make Ex 2.1 fully author-driven:** Pre-select two crime types (e.g. Larceny Theft and Prostitution), add text annotations at their peaks ('Larceny peaks at 2pm — retail hours'; 'Prostitution peaks at midnight'), disable legend interactivity, and add a paragraph of text before the chart explaining what to notice.

## Exercise 2.1 — Micro-Project Data Story

The data story is published on the GitHub Pages site. Below is the content and the code used to generate its three visualizations.

**Story: 'The Quietest Crisis — How Drug Enforcement Quietly Disappeared from San Francisco's Data'**

This story examines the dramatic, sustained decline in drug offense arrests in San Francisco — and asks what it actually means.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, warnings, requests
warnings.filterwarnings('ignore')

# Load pre-merged dataset (from Week 2)
try:
    merged = pd.read_parquet('/home/claude/notebooks/sf_crime_merged.parquet')
    print(f'Loaded merged dataset: {len(merged):,} rows')
except FileNotFoundError:
    print('Run Week 2 first to generate the merged parquet, or re-fetch here.')
    # Fallback: re-fetch just 'now' data for standalone use
    import requests
    API_NOW = 'https://data.sfgov.org/resource/wg3w-h783.json'
    def fetch_all(url, params=None, chunk=50_000):
        if params is None: params = {}
        records, offset = [], 0
        while True:
            r = requests.get(url, params={**params, '$limit': chunk, '$offset': offset}, timeout=120)
            r.raise_for_status(); batch = r.json()
            if not batch: break
            records.extend(batch); offset += chunk
            if len(batch) < chunk: break
        return pd.DataFrame(records)
    raw = fetch_all(API_NOW)
    raw['date'] = pd.to_datetime(raw.get('incident_date', pd.NaT), errors='coerce')
    raw['year'] = raw['date'].dt.year
    raw['month'] = raw['date'].dt.month
    raw['weekday'] = raw['date'].dt.day_name()
    raw['hour'] = pd.to_datetime(raw.get('incident_time',''), format='%H:%M', errors='coerce').dt.hour
    raw['category'] = raw.get('incident_category', '')
    raw['district'] = raw.get('police_district', '')
    raw['lat'] = pd.to_numeric(raw.get('latitude', None), errors='coerce')
    raw['lon'] = pd.to_numeric(raw.get('longitude', None), errors='coerce')
    merged = raw.dropna(subset=['date']).copy()

FOCUS_CRIMES = [
    'Assault', 'Burglary', 'Drug Offense', 'Larceny Theft',
    'Motor Vehicle Theft', 'Robbery', 'Vandalism', 'Weapons Offense',
    'Arson', 'Disorderly Conduct', 'Fraud', 'Prostitution',
    'Sex Offense', 'Stolen Property', 'Suspicious Occ', 'Traffic Violation Arrest'
]
FOCUS_CRIMES = [c for c in FOCUS_CRIMES if c in merged['category'].unique()]
df_focus = merged[merged['category'].isin(FOCUS_CRIMES)].copy()
print(f'Focus crimes available: {len(FOCUS_CRIMES)}')


### Figure 1 (Static): Drug Offense long-term trend vs Larceny Theft

In [ ]:
import matplotlib.pyplot as plt, matplotlib.ticker as mticker

drug   = merged[merged['category']=='Drug Offense'].groupby('year').size()
larceny = merged[merged['category']=='Larceny Theft'].groupby('year').size()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

l1, = ax1.plot(drug.index, drug.values,   color='#e94560', linewidth=2.5, marker='o', markersize=5, label='Drug Offense')
l2, = ax2.plot(larceny.index, larceny.values, color='#2196f3', linewidth=2.5, marker='s', markersize=5, linestyle='--', label='Larceny Theft')

ax1.set_ylabel('Drug Offense arrests', color='#e94560', fontsize=11)
ax2.set_ylabel('Larceny Theft incidents', color='#2196f3', fontsize=11)
ax1.tick_params(axis='y', colors='#e94560')
ax2.tick_params(axis='y', colors='#2196f3')
ax1.set_xlabel('Year', fontsize=11)
ax1.set_title('Drug Offense Arrests vs. Larceny Theft in San Francisco (2003–present)',
              fontweight='bold', fontsize=13)

# Annotate key events
ax1.axvline(2014, color='gray', linestyle=':', alpha=0.7)
ax1.text(2014.2, drug.max()*0.92, 'Prop 47\n(2014)', fontsize=8, color='gray')
ax1.axvline(2020, color='gray', linestyle=':', alpha=0.7)
ax1.text(2020.2, drug.max()*0.92, 'COVID\n(2020)', fontsize=8, color='gray')

ax1.legend(handles=[l1, l2], loc='upper right')
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('story_fig1_drug_vs_larceny.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved story_fig1_drug_vs_larceny.png')


**Figure 1 caption:** Drug offense arrests in San Francisco have fallen by more than 80% since their peak in the mid-2000s (red line, left axis). This decline accelerated sharply after Proposition 47 (2014), which reclassified many drug possession offenses from felonies to misdemeanors, reducing incentives for arrest. Larceny Theft (blue dashed line, right axis) shows no comparable long-term decline — confirming that the drug enforcement drop reflects *policy change*, not a general crime reduction.

### Figure 2 (Map): Drug Offense geographic concentration

In [ ]:
import folium
from folium.plugins import HeatMap

drug_geo = merged[(merged['category']=='Drug Offense')].dropna(subset=['lat','lon'])
drug_geo = drug_geo[(drug_geo['lat'].between(37.70, 37.83)) &
                    (drug_geo['lon'].between(-122.53, -122.35))]

m = folium.Map([37.7749, -122.4194], zoom_start=13, tiles='OpenStreetMap')
HeatMap(drug_geo[['lat','lon']].values.tolist(), radius=12, blur=8, min_opacity=0.3).add_to(m)
folium.Marker([37.7849, -122.4094], popup='Tenderloin District',
              icon=folium.Icon(color='red')).add_to(m)
m.save('story_fig2_drug_map.html')
print(f'Saved story_fig2_drug_map.html ({len(drug_geo):,} points)')
m


**Figure 2 caption:** Heatmap of all Drug Offense arrests across San Francisco (2003–present). The concentration in the Tenderloin district (red marker) is extreme — a single neighborhood accounts for a disproportionate share of all arrests. This geographic concentration raises the question: does the data show where *drug use* is highest, or where *police enforcement* was most intensive?

### Figure 3 (Interactive): Drug Offense arrests by district over time

In [ ]:
import plotly.express as px

drug_dist = merged[(merged['category']=='Drug Offense') &
                   merged['district'].notna()].groupby(['year','district']).size().reset_index(name='count')

fig = px.line(drug_dist, x='year', y='count', color='district',
              title='Drug Offense Arrests by Police District (2003–present)',
              labels={'count': 'Arrests', 'year': 'Year', 'district': 'Police District'},
              markers=True)

fig.update_traces(line=dict(width=2))
fig.update_layout(
    height=500,
    legend=dict(orientation='v', x=1.01, y=1),
    xaxis=dict(dtick=2),
    template='plotly_dark'
)

# Highlight Tenderloin
for trace in fig.data:
    if trace.name and 'TENDERLOIN' in trace.name.upper():
        trace.line.width = 4
        trace.line.color = '#e94560'

fig.write_html('story_fig3_drug_districts.html')
fig.show()
print('Saved story_fig3_drug_districts.html')


**Figure 3 caption:** Drug offense arrest counts by police district, 2003–present. The Tenderloin (highlighted in red) consistently accounts for the largest share of arrests. Click on district names in the legend to show or hide them. Notice that *all* districts show declining arrests — this is not just a Tenderloin phenomenon, but a citywide shift in enforcement priorities.

## Exercise 3.1 — Reflection

**Hardest part:** The writing. It's easy to describe patterns ('drug arrests went down') but hard to interpret them responsibly ('this reflects policy change, not drug use change'). The risk of implying causation where there is only correlation is constant.

**Author-driven vs reader-driven tension:** The story is author-driven — we chose what to show and in what order. But Figure 3's interactivity gives the reader agency to explore district-level differences. This tension works in our favor: the interactive chart supports our narrative argument while letting skeptical readers verify the pattern in their own preferred district.

**If I had unlimited time:** I would add scrollytelling — text that animates the chart as the user scrolls, highlighting different districts at different narrative moments. I would also add a comparison with self-reported drug use data from city surveys, to directly test whether the arrest decline reflects actual behavior change or purely enforcement change.